# 02_rt08 — 기존 정의 R0의 v2 재학습 (RT-08)

[최종 실험 계획서](../../../docs/ml/ROUTER_FINAL_EXPERIMENT_PLAN.md) 3절. 운영 R0(RT-04)와 **같은 학습 함수·설정·3 epoch**로, 라벨만 v2 채점 기준(전체 모델 해결률, 경계 .27/.64)으로 바꿔 다시 학습한다. test는 읽지 않는다(08_test에서 한 번).

**런타임: 2026.07(Python 3.12), T4 GPU.** 약 1시간. 끊기면 다시 실행하면 저장된 지점부터 이어서 한다.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
from pathlib import Path
import hashlib, json, os, subprocess, sys, zipfile
RUN = Path('/content/drive/MyDrive/LLM_Router/router-final/runs/run001')
SOURCE_ZIP = RUN / 'code' / 'router-final-source.zip'
expected = SOURCE_ZIP.with_suffix('.sha256').read_text().split()[0]
assert hashlib.sha256(SOURCE_ZIP.read_bytes()).hexdigest() == expected, '소스 ZIP이 바뀌었다'
REPO = Path('/content/router-final-source')
with zipfile.ZipFile(SOURCE_ZIP) as z:
    for name in z.namelist():
        assert (REPO / name).resolve().is_relative_to(REPO.resolve()), name
    z.extractall(REPO)
for name, digest in json.loads((REPO / 'source-bundle-manifest.json').read_text()).items():
    assert hashlib.sha256((REPO / name).read_bytes()).hexdigest() == digest, name
assert sys.version_info[:2] in {(3, 11), (3, 12)}, '런타임 버전을 2026.07(Python 3.12)로 바꾸세요'
subprocess.run([sys.executable, '-m', 'pip', 'uninstall', '-y', 'torchvision', 'torchaudio'], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', 'torch==2.5.1', '--index-url', 'https://download.pytorch.org/whl/cu124'], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-r', str(REPO / 'ml/notebooks/router_final/requirements-train-colab.txt')], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '--no-deps', '-e', str(REPO / 'ml')], check=True)

설치가 끝나면 **런타임 → 세션 다시 시작**을 한 번 한 뒤, 아래 셀부터 실행합니다(새로 설치한 torch를 불러오기 위해).

In [ ]:
from pathlib import Path
import json, os, subprocess, sys
RUN = Path('/content/drive/MyDrive/LLM_Router/router-final/runs/run001')
REPO = Path('/content/router-final-source')
import torch
assert torch.cuda.is_available(), 'T4 GPU 런타임을 선택하세요'
print(torch.cuda.get_device_name(0), torch.__version__, torch.version.cuda)
ENV = dict(os.environ, PYTHONPATH=str(REPO / 'ml/src'))
RT08 = REPO / 'ml/notebooks/router_final/rt08.py'
DATA = RUN / 'rt08' / 'data'
OUT = RUN / 'rt08' / 'model'
OUT.mkdir(parents=True, exist_ok=True)

In [ ]:
# 1) 40 step 시험 학습 (몇 분)
subprocess.run([sys.executable, str(RT08), 'train', '--root', str(REPO), '--data-dir', str(DATA), '--persistent-dir', str(OUT), '--pilot'], env=ENV, check=True)

In [ ]:
# 2) 본 학습: 3 epoch, 1,992 step (약 1시간)
subprocess.run([sys.executable, str(RT08), 'train', '--root', str(REPO), '--data-dir', str(DATA), '--persistent-dir', str(OUT)], env=ENV, check=True)
print(sorted(p.name for p in (OUT / 'full').iterdir()))